In [ ]:
from google.colab import drive

drive.mount("/content/drive")


# Coleta do Plenario do Senado

Notebook Colab para clonar o repositorio, instalar dependencias e executar o coletor `coleta.senado.plenario_discursos.collect` salvando dados no Google Drive.

A primeira celula monta o Drive antes de qualquer codigo do projeto. Os dados completos ficam em `MyDrive/falando_nela/data`; o clone do repositorio fica em `/content/falando_nela` e pode ser recriado a cada sessao.


In [ ]:
import os
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data")
os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)

for name in ["raw", "checkpoints", "logs", "manifests", "processed"]:
    (DATA_ROOT / name).mkdir(parents=True, exist_ok=True)

print("FALANDO_NELA_DATA_ROOT=", os.environ["FALANDO_NELA_DATA_ROOT"])


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_DIR = Path("/content/falando_nela")
REPO_REF = ""  # opcional: branch, tag ou commit. Deixe vazio para usar o default remoto.

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], check=True)
os.chdir(REPO_DIR)
print("Repo em:", Path.cwd())
subprocess.run(["git", "status", "--short"], check=True)


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


## Parametros

Use a validacao curta primeiro. A coleta completa esta protegida por uma flag para evitar rodar o baseline inteiro sem querer.


In [ ]:
from datetime import datetime, timezone

DATA_INICIO_VALIDACAO = "2011-05-18"
DATA_FIM_VALIDACAO = "2011-05-31"
SAMPLE_LIMIT_VALIDACAO = 10

DATA_INICIO_PROD = "2011-05-18"
DATA_FIM_PROD = "2026-05-18"

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ID_VALIDACAO = f"colab-senado-plenario-validacao-{RUN_STAMP}"
RUN_ID_PROD = "prod-senado-plenario-baseline"

print("RUN_ID_VALIDACAO=", RUN_ID_VALIDACAO)
print("RUN_ID_PROD=", RUN_ID_PROD, "# fixo para permitir retomada com --resume")


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

MODULE = "coleta.senado.plenario_discursos.collect"


def run_collector(run_id, data_inicio, data_fim, sample_limit=None, resume=True, stream=True):
    cmd = [
        sys.executable, "-m", MODULE,
        "--mode", "prod",
        "--run-id", run_id,
        "--data-inicio", data_inicio,
        "--data-fim", data_fim,
    ]
    if resume:
        cmd.append("--resume")
    if sample_limit is not None:
        cmd.extend(["--sample-limit", str(sample_limit)])

    print("Rodando:", " ".join(cmd))
    output_lines = []
    if stream:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
            output_lines.append(line.rstrip("\n"))
        returncode = process.wait()
    else:
        result = subprocess.run(cmd, check=False, text=True, capture_output=True)
        returncode = result.returncode
        if result.stdout:
            print(result.stdout)
            output_lines.extend(result.stdout.splitlines())
        if result.stderr:
            print(result.stderr)
            output_lines.extend(result.stderr.splitlines())

    manifest_path = manifest_from_output(output_lines, run_id)
    if returncode != 0:
        print(f"Coletor terminou com returncode={returncode}; a celula continuara para permitir inspecao de logs/autosave.")
    return manifest_path


def manifest_from_output(output_lines, run_id):
    for line in reversed(output_lines):
        candidate = line.strip()
        if candidate.endswith(".json"):
            path = Path(candidate)
            if path.exists():
                return path
    manifest_path = DATA_ROOT / "manifests" / f"{run_id}.json"
    if manifest_path.exists():
        return manifest_path
    autosave_path = DATA_ROOT / "manifests" / f"{run_id}.autosave.json"
    if autosave_path.exists():
        return autosave_path
    return None


def show_manifest(manifest_path):
    if manifest_path is None:
        print("Manifest nao foi informado pelo coletor.")
        return None
    manifest_path = Path(manifest_path)
    if not manifest_path.exists():
        print("Manifest nao encontrado:", manifest_path)
        return None
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    resumo = {
        "run_id": manifest.get("run_id"),
        "source": manifest.get("source"),
        "dataset": manifest.get("dataset"),
        "mode": manifest.get("mode"),
        "sample": manifest.get("sample"),
        "sample_limit": manifest.get("sample_limit"),
        "status": manifest.get("status"),
        "errors": manifest.get("errors"),
        "record_counts": manifest.get("record_counts"),
        "partition_counts": manifest.get("partition_counts"),
        "processed_records_loaded": manifest.get("processed_records_loaded"),
        "log_path": manifest.get("log_path"),
        "autosave_path": manifest.get("autosave_path"),
    }
    print(json.dumps(resumo, ensure_ascii=False, indent=2))

    log_path = Path(manifest["log_path"])
    if log_path.exists():
        print("\nUltimas linhas do log:")
        print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-10:]))
    return manifest


def tail_log_for_run(run_id, n=30):
    log_path = DATA_ROOT / "logs" / f"{run_id}.jsonl"
    print("log_path=", log_path)
    if not log_path.exists():
        print("Log ainda nao encontrado.")
        return
    print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-n:]))


## Validacao curta

Esta celula roda em `--mode prod`, mas limita a janela e o numero de textos. Ela confirma que o Drive esta montado, que a escrita esta indo para `FALANDO_NELA_DATA_ROOT` e que o endpoint de texto integral esta retornando conteudo.


In [ ]:
manifest_validacao_path = run_collector(
    RUN_ID_VALIDACAO,
    DATA_INICIO_VALIDACAO,
    DATA_FIM_VALIDACAO,
    sample_limit=SAMPLE_LIMIT_VALIDACAO,
    resume=True,
)
manifest_validacao = show_manifest(manifest_validacao_path)


In [ ]:
def inspect_jsonl(path, max_rows=5):
    path = Path(path)
    print("\nArquivo:", path)
    if not path.exists():
        print("Nao encontrado")
        return
    lines = path.read_text(encoding="utf-8").splitlines()
    print("linhas=", len(lines), "bytes=", path.stat().st_size)
    for line in lines[:max_rows]:
        record = json.loads(line)
        payload = record.get("payload", {})
        print({
            "record_type": record.get("record_type"),
            "partition": record.get("partition"),
            "CodigoPronunciamento": payload.get("CodigoPronunciamento"),
            "metodo_obtencao": payload.get("metodo_obtencao"),
            "tem_TextoIntegral": bool(payload.get("TextoIntegral")),
        })

metadata_path = DATA_ROOT / "raw" / "senado" / "plenario_discursos" / "metadata" / f"{RUN_ID_VALIDACAO}.jsonl"
corpus_path = DATA_ROOT / "raw" / "senado" / "plenario_discursos" / "ano=2011" / "mes=05" / f"{RUN_ID_VALIDACAO}.jsonl"

inspect_jsonl(metadata_path, max_rows=2)
inspect_jsonl(corpus_path, max_rows=5)


## Coleta de madrugada

Esta celula roda o baseline inteiro em modo producao, com `run_id` fixo e `--resume`. Se o Colab cair ou a API falhar em alguma parte, rode a mesma celula novamente: o coletor le o checkpoint e os JSONLs ja gravados antes de continuar.


In [ ]:
EXECUTAR_MADRUGADA = True
RUN_ID_MADRUGADA = RUN_ID_PROD

if EXECUTAR_MADRUGADA:
    print("Rodando coleta de madrugada com run_id fixo:", RUN_ID_MADRUGADA)
    manifest_madrugada_path = run_collector(
        RUN_ID_MADRUGADA,
        DATA_INICIO_PROD,
        DATA_FIM_PROD,
        sample_limit=None,
        resume=True,
        stream=True,
    )
    manifest_madrugada = show_manifest(manifest_madrugada_path)
else:
    print("Coleta de madrugada nao executada. Defina EXECUTAR_MADRUGADA = True para rodar.")


## Inspecao e retomada

Use esta celula na manha seguinte, ou depois de uma interrupcao, para ver o autosave e as ultimas linhas do log. Para continuar, rode novamente a celula de coleta de madrugada com o mesmo `RUN_ID_MADRUGADA`.


In [ ]:
autosave_madrugada_path = DATA_ROOT / "manifests" / f"{RUN_ID_MADRUGADA}.autosave.json"
manifest_madrugada_path = DATA_ROOT / "manifests" / f"{RUN_ID_MADRUGADA}.json"

if manifest_madrugada_path.exists():
    show_manifest(manifest_madrugada_path)
elif autosave_madrugada_path.exists():
    show_manifest(autosave_madrugada_path)
else:
    print("Ainda nao ha manifest/autosave para", RUN_ID_MADRUGADA)

tail_log_for_run(RUN_ID_MADRUGADA, n=30)
